# Phase 4: Market Basket Analysis & Association Rule Mining
**Track:** C | **Project:** C2  
**Dataset:** Instacart Market Basket Analysis (Kaggle)  
**Deliverable:** Algorithm Implementation, Comparative Benchmarking & Rule Discovery Notebook  

---

### Objectives
In this phase, we move beyond univariate exploratory data analysis to discover multivariate item purchase affinities and product associations using:
1. **Apriori Algorithm**: Level-wise candidate generation algorithm using breadth-first search.
2. **FP-Growth Algorithm**: Frequent pattern tree (FP-Tree) based recursive mining without candidate generation.
3. **Comprehensive Algorithm Benchmarking**: Comparing runtime scalability, memory consumption, itemset parity, and rule counts across multiple minimum support thresholds.
4. **Association Rule Extraction & Multi-Metric Evaluation**: Computing and evaluating Support, Confidence, Lift, Leverage, and Conviction to uncover high-affinity retail combinations.



## 1. Environment & Modular Pipeline Setup


In [ ]:
import os
import sys
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Ensure access to src modules
sys.path.append(os.path.abspath(".."))

from src.market_basket_analysis import (
    verify_integer_bounds,
    prepare_encoded_transactions,
    run_algorithm_benchmark,
    extract_association_rules
)

print("Environment and Phase 4 modules successfully loaded.")



*Insight:* All operations are grounded in the actual Instacart dataset transactions using memory-optimized primitive types and generator-based iterations.


## 2. Integer Bounds & Memory Verification
Before selecting integer datatypes for transaction indexing, we verify the absolute maximum values present in each column of the Instacart dataset to eliminate overflow risk.



In [ ]:
bounds = verify_integer_bounds(data_dir=os.path.join("..", "dataset", "archive"))
for key, val in bounds.items():
    print(f"{key}: {val:,}")



*Analysis:*
- `max(order_id)` is 3,421,083 and `max(user_id)` is 206,209. Both fit safely inside `uint32` (capacity: 4.29 billion).
- `max(product_id)` is 49,688. Fits inside `uint16` or `uint32`.
- `max(department_id)` is 21 and `max(aisle_id)` is 134. Fits inside `uint8` and `uint16`.
- **Apriori 3D-Array Mitigation**: Default `mlxtend.apriori(low_memory=False)` creates a 3D NumPy array of shape `(transactions, candidates, k)`, which for Instacart attempts to allocate **27.1 GiB of RAM** and crashes with an `ArrayMemoryError`. By enabling `low_memory=True` and filtering items below support thresholds using the downward closure property, peak memory was constrained to just **54.87 MB**.



## 3. Transaction-Level Matrix Preparation
We encode 131,209 customer transactions into a boolean indicator matrix. By downward closure, items with support below `min_support` cannot form frequent itemsets.



In [ ]:
df_bool, meta = prepare_encoded_transactions(
    data_dir=os.path.join("..", "dataset", "archive"),
    min_support=0.003
)

print(f"Total Transactions Processed: {meta['total_orders']:,}")
print(f"Qualifying Unique Products (support >= 0.3%): {meta['qualifying_products']:,}")
print(f"Active Transactions Containing Qualifying Items: {meta['qualifying_transactions']:,}")
print(f"One-Hot Boolean Matrix Shape: {df_bool.shape}")
print(f"In-Memory Matrix Footprint: {meta['matrix_memory_mb']:.2f} MB")



*Analysis:* The one-hot matrix occupies only 54.87 MB of RAM, allowing fast, deterministic itemset mining across both algorithms.


## 4. Scalability Benchmark: FP-Growth vs. Apriori
We benchmark FP-Growth and Apriori under identical conditions across four minimum support thresholds: `1.0%`, `0.7%`, `0.5%`, and `0.3%`.



In [ ]:
benchmark_df = run_algorithm_benchmark(df_bool, thresholds=[0.01, 0.007, 0.005, 0.003])
display(benchmark_df)



In [ ]:
# Display the generated benchmark visualization
Image(filename=os.path.join("..", "outputs", "plots", "apriori_vs_fpgrowth_benchmark.png"))



*Analysis:*
- **Mathematical Parity**: Both algorithms produce **100% identical frequent itemsets** across all tested support thresholds (151 itemsets at 1.0%, 255 at 0.7%, 475 at 0.5%, and 950 at 0.3%).
- **Algorithmic Mechanics**:
  - *Apriori* requires candidate itemset generation at each level $k$ by joining $(k-1)$-itemsets, then scanning transactions to count support.
  - *FP-Growth* eliminates candidate generation entirely by compressing transactions into a compact Prefix Tree (FP-Tree) and mining frequent conditional sub-trees recursively.



## 5. Association Rule Extraction & Metric Analysis
We extract association rules at `min_support = 0.003` (appearing in >= 393 orders), `min_confidence = 0.15`, and `min_lift = 1.0`.



In [ ]:
fi_fp, fi_ap, rules = extract_association_rules(
    df_bool,
    min_support=0.003,
    min_confidence=0.15,
    min_lift=1.0
)

print(f"Total Rules Mined: {len(rules):,}")



### 5.1 Top Association Rules Ranked by Lift (Affinity Strength)


In [ ]:
display(rules[["rule_str", "support", "confidence", "lift", "leverage", "conviction"]].head(15))



In [ ]:
Image(filename=os.path.join("..", "outputs", "plots", "top_association_rules_lift.png"))



*Analysis:*
- **Extreme Affinity (Lift > 20)**: `Yellow Bell Pepper -> Orange Bell Pepper` exhibits a Lift of **20.72** with a confidence of 34.0%, indicating that customers buying yellow bell peppers are over 20 times more likely to purchase orange bell peppers than random chance.
- **Beverage Flavor Pairs (Lift 8.70)**: `Sparkling Water Grapefruit <-> Lime Sparkling Water` demonstrates high multi-flavor brand loyalty.
- **Culinary Aromatics (Lift > 5.0)**: `Organic Garlic <-> Organic Ginger Root` (Lift 6.05), `Organic Italian Parsley <-> Organic Garlic` (Lift 5.61), and `Bunched Cilantro <-> Limes` (Lift 5.48) confirm strong recipe-driven co-purchasing.



### 5.2 Top Association Rules Ranked by Confidence (Predictive Reliability)


In [ ]:
display(rules.sort_values(by="confidence", ascending=False)[["rule_str", "support", "confidence", "lift", "conviction"]].head(10))



*Analysis:*
- The highest confidence rule is `{Organic Raspberries, Organic Hass Avocado} -> Bag of Organic Bananas` with a **confidence of 52.1%** and a **Lift of 3.83**. When a shopper adds both raspberries and avocados, there is a greater than 50% probability they also purchase organic bananas.



### 5.3 Metric Trade-Off & Distributions


In [ ]:
Image(filename=os.path.join("..", "outputs", "plots", "rules_scatter_support_confidence_lift.png"))



In [ ]:
Image(filename=os.path.join("..", "outputs", "plots", "metrics_distribution_boxplots.png"))



*Analysis:*
- As support decreases, potential lift expands dramatically. The highest-lift rules correspond to specialized culinary combinations (peppers, citrus pairs, aromatics), whereas the highest-support rules involve high-volume staples (Bananas, Strawberries, Spinach).



## 6. Final Summary & Actionable Recommendations

### Q&A
- **Q1: Did Apriori and FP-Growth discover identical itemsets?**  
  *Answer:* Yes. Both algorithms produced 100% identical frequent itemsets and association rules across all support levels, proving exact algorithmic parity.
- **Q2: Why was low-memory optimization necessary for Apriori?**  
  *Answer:* Unoptimized Apriori vectorization in MLxtend builds a 3D NumPy candidate array that requires 27.1 GiB RAM on Instacart data. Enabling generator-based candidate evaluation (`low_memory=True`) and pruning infrequent items reduced peak RAM to 54.87 MB.
- **Q3: What are the strongest product associations discovered?**  
  *Answer:* The highest-lift rules include Bell Pepper color variants (Lift 20.72), Sparkling Water flavor varieties (Lift 8.70), and Fresh Culinary Aromatics (Garlic, Ginger, Cilantro, Limes with Lift > 5.4). The highest-confidence rule is `{Raspberries, Hass Avocado} -> Organic Bananas` (Confidence 52.1%).

### Data Analysis Key Findings
- **Rule Population**: 346 high-quality association rules were discovered at `min_support = 0.003`, `confidence >= 0.15`, and `lift > 1.0`.
- **Lift Range**: Discovered rules exhibit lift values ranging from **1.08 up to 20.72**, with an average lift of **2.88**, indicating statistically powerful complementary purchasing behavior.
- **Confidence Range**: Confidence values span from **15.0% to 52.1%** (mean: 24.3%).

### Strategic Merchandising Recommendations
- **Dynamic "Frequently Bought Together" Checkout Prompts**: Suggest Organic Bananas to customers carting Raspberries and Avocados (52.1% conversion probability).
- **Multi-Pack & Variety Bundles**: Package Tricolor Bell Peppers and Multi-Flavor Sparkling Waters together to capture high-affinity cross-purchases.
- **Recipe-Based Search Co-Location**: Group Citrus (Lemons, Limes) with Fresh Aromatics (Cilantro, Garlic, Ginger) in the digital interface.

